In [52]:
import sys
import io
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
import json
import os
import sys
sys.path.insert(0, r"C:\Users\admin\work\idf_project\notebooks")
import config as cfg
from sqlalchemy import create_engine, text

engine = create_engine(cfg.DB_URL)

print("Loading data from PostGIS...")

# Load commune index with geometry
index_gdf = gpd.read_postgis("""
    SELECT 
        ci.insee_com,
        ci.nom_com,
        ci.eai_score,
        ci.eai_rank,
        ci.score_food_access,
        ci.score_health_core,
        ci.score_health_complementary,
        ci.score_education_primary,
        ci.score_education_secondary,
        ci.score_education_higher,
        ci.score_social_services,
        ci.score_transport,
        ci.score_sports,
        ci.score_culture,
        ci.score_public_services,
        s.med_income_2021,
        s.poverty_rate,
        s.pop_total,
        s.pct_65plus,
        s.pct_0014,
        ci.geom AS geometry
    FROM idf.commune_index ci
    LEFT JOIN idf.socioeco s ON ci.insee_com = s.insee_com
""", engine, geom_col='geometry')

# Force correct CRS and reproject to WGS84
index_gdf = index_gdf.set_crs(epsg=2154, allow_override=True).to_crs(epsg=4326)

print(f"Loaded {len(index_gdf)} communes")
print(f"CRS: {index_gdf.crs}")
print(f"Bounds: {index_gdf.total_bounds}")

# Load facilities
facilities_gdf = gpd.read_postgis("""
    SELECT typequ, typequ_label, indicator, dep, insee_com, geom AS geometry
    FROM idf.facilities
""", engine, geom_col='geometry')

facilities_gdf = facilities_gdf.set_crs(epsg=2154, allow_override=True).to_crs(epsg=4326)
print(f"Loaded {len(facilities_gdf)} facilities")

Loading data from PostGIS...
Loaded 1266 communes
CRS: EPSG:4326
Bounds: [ 1.44630006 48.12007725  3.55902711 49.24148776]
Loaded 89208 facilities


In [53]:
LAYERS = {
    'eai_score':                    'IPS - Score',
    'score_health_core':            'Soins essentiels',
    'score_health_complementary':   'Soins complémentaires',
    'score_food_access':            'Alimentaire',
    'score_transport':              'Transports',
    'score_education_primary':      'Éducation — Premier degré',
    'score_education_secondary':    'Éducation — Second degré',
    'score_education_higher':       'Éducation — Supérieur',
    'score_social_services':        'Services sociaux',
    'score_sports':                 'Sports',
    'score_culture':                'Culture',
    'score_public_services':        'Sécurité',
}

FACILITY_GROUPS = {
    'Soins essentiels': [
        'health_gp', 'health_pharmacy', 'health_urgences'
    ],
    'Soins complémentaires': [
        'health_centre', 'health_msp', 'health_dental',
        'health_nurse', 'health_lab', 'health_maternity', 'health_pmi'
    ],
    'Éducation': [
        'school_maternelle', 'school_primary', 'college',
        'lycee_general', 'lycee_pro', 'higher_public', 'higher_private',
        'vocational_training'
    ],
    'Alimentaire': [
        'supermarket_large', 'local_food_shop', 'boulangerie'
    ],
    'Transports': [
        'train_national', 'train_regional', 'train_local'
    ],
    'Services sociaux': [
        'elderly_care', 'childcare', 'childcare_loisir', 'social_centre'
    ],
    'Sports': [
        'sport_pool', 'sport_outdoor', 'sport_indoor', 'sport_other'
    ],
    'Culture': ['culture'],
    'Sécurité': ['security'],
}

FACILITY_COLORS = {
    'Soins essentiels':       'red',
    'Soins complémentaires':  'pink',
    'Éducation':              'blue',
    'Alimentaire':            'orange',
    'Transports':             'darkblue',
    'Services sociaux':       'purple',
    'Sports':                 'green',
    'Culture':                'darkred',
    'Sécurité':               'gray',
}

print(f"Catégories carte : {len(LAYERS)}")
print(f"Groupes équipements : {len(FACILITY_GROUPS)}")

Catégories carte : 12
Groupes équipements : 9


In [54]:
def build_tooltip_html(row):

    def score_color(score):
        if score is None or pd.isna(score): return '#cccccc'
        if score >= 0.75: return '#1a9641'
        elif score >= 0.50: return '#a6d96a'
        elif score >= 0.25: return '#fdae61'
        else: return '#d7191c'

    def fmt_score(score):
        if score is None or pd.isna(score): return 'N/A'
        return f"{score:.3f}"

    def fmt_currency(val):
        if val is None or pd.isna(val): return 'N/A'
        return f"{int(val):,}&euro;".replace(',', ' ')

    def fmt_pct(val):
        if val is None or pd.isna(val): return 'N/A'
        return f"{val:.1f}%"

    dims = {
        'Soins essentiels':     row.get('score_health_core'),
        'Soins complémentaires':row.get('score_health_complementary'),
        'Alimentaire':          row.get('score_food_access'),
        'Transports':           row.get('score_transport'),
        'Éducation primaire':   row.get('score_education_primary'),
        'Éducation secondaire': row.get('score_education_secondary'),
        'Éducation supérieure': row.get('score_education_higher'),
        'Services sociaux':     row.get('score_social_services'),
        'Sports':               row.get('score_sports'),
        'Culture':              row.get('score_culture'),
        'Sécurité':             row.get('score_public_services'),
    }

    dim_rows = ""
    for dim_name, dim_score in dims.items():
        color = score_color(dim_score)
        bar_width = int((dim_score or 0) * 100)
        dim_rows += f"""
        <tr>
            <td style="padding:2px 6px;font-size:11px;white-space:nowrap">{dim_name}</td>
            <td style="padding:2px 6px;width:80px">
                <div style="background:#eee;border-radius:2px;height:10px">
                    <div style="background:{color};width:{bar_width}%;height:10px;border-radius:2px"></div>
                </div>
            </td>
            <td style="padding:2px 6px;font-size:11px;color:{color};font-weight:bold">
                {fmt_score(dim_score)}
            </td>
        </tr>"""

    pop = row.get('pop_total')
    pop_str = f"{int(pop):,}".replace(',', ' ') if pd.notna(pop) else 'N/A'
    rank = row.get('eai_rank')
    rank_str = f"{int(rank)}" if pd.notna(rank) else 'N/A'

    html = f"""
    <meta charset="utf-8">
    <div style="font-family:Arial,sans-serif;min-width:280px;max-width:320px">
        <div style="background:#2c3e50;color:white;padding:10px 12px;
                    border-radius:6px 6px 0 0">
            <div style="font-size:15px;font-weight:bold">{row['nom_com']}</div>
            <div style="font-size:11px;opacity:0.8">INSEE : {row['insee_com']}</div>
        </div>
        <div style="background:{score_color(row.get('eai_score'))};color:white;
                    padding:8px 12px;display:flex;justify-content:space-between;
                    align-items:center">
            <span style="font-size:12px;font-weight:bold">Score IPS</span>
            <span style="font-size:20px;font-weight:bold">{fmt_score(row.get('eai_score'))}</span>
            <span style="font-size:12px">Rang #{rank_str}/1 266</span>
        </div>
        <div style="padding:8px 12px;background:white">
            <div style="font-size:11px;font-weight:bold;color:#666;
                        margin-bottom:4px;text-transform:uppercase;letter-spacing:0.5px">
                Scores par dimension
            </div>
            <table style="width:100%;border-collapse:collapse">
                {dim_rows}
            </table>
        </div>
        <div style="padding:8px 12px;background:#f8f9fa;
                    border-top:1px solid #dee2e6;border-radius:0 0 6px 6px">
            <div style="font-size:11px;font-weight:bold;color:#666;
                        margin-bottom:4px;text-transform:uppercase;letter-spacing:0.5px">
                Contexte socio-économique
            </div>
            <table style="width:100%;border-collapse:collapse">
                <tr>
                    <td style="font-size:11px;padding:2px 6px;color:#444">Revenu médian</td>
                    <td style="font-size:11px;padding:2px 6px;font-weight:bold;text-align:right">
                        {fmt_currency(row.get('med_income_2021'))}
                    </td>
                </tr>
                <tr>
                    <td style="font-size:11px;padding:2px 6px;color:#444">Taux de pauvreté</td>
                    <td style="font-size:11px;padding:2px 6px;font-weight:bold;text-align:right">
                        {fmt_pct(row.get('poverty_rate'))}
                    </td>
                </tr>
                <tr>
                    <td style="font-size:11px;padding:2px 6px;color:#444">Population</td>
                    <td style="font-size:11px;padding:2px 6px;font-weight:bold;text-align:right">
                        {pop_str}
                    </td>
                </tr>
                <tr>
                    <td style="font-size:11px;padding:2px 6px;color:#444">Part 65+ (%)</td>
                    <td style="font-size:11px;padding:2px 6px;font-weight:bold;text-align:right">
                        {fmt_pct(row.get('pct_65plus'))}
                    </td>
                </tr>
                <tr>
                    <td style="font-size:11px;padding:2px 6px;color:#444">Part 0-14 (%)</td>
                    <td style="font-size:11px;padding:2px 6px;font-weight:bold;text-align:right">
                        {fmt_pct(row.get('pct_0014'))}
                    </td>
                </tr>
            </table>
        </div>
        <div style="font-size:9px;color:#999;padding:4px 12px;text-align:center">
            Score = rang percentile parmi les 1 266 communes d'Île-de-France.<br>
            1,0 = mieux équipée &nbsp;·&nbsp; 0,0 = moins bien équipée.
        </div>
    </div>
    """
    return html

print("Tooltip prêt")

Tooltip prêt


In [55]:
# Build popup lookup — keyed by insee_com
# We inject these via JavaScript rather than GeoJSON properties
# to avoid JSON escaping issues
popup_lookup = {}
for _, row in index_gdf.iterrows():
    popup_lookup[row['insee_com']] = build_tooltip_html(row)

print(f"Built {len(popup_lookup)} popups")
print(f"Sample popup length: {len(popup_lookup['75056'])} chars")

Built 1266 popups
Sample popup length: 9020 chars


In [56]:
# ── Reprojection ──────────────────────────────────────────────────────────────
print("Construction de la carte...")
geojson_data = json.loads(index_gdf.to_json())

score_cols = list(LAYERS.keys())

for feature in geojson_data['features']:
    insee = feature['properties']['insee_com']
    rows = index_gdf[index_gdf['insee_com'] == insee]
    if len(rows) == 0:
        continue
    row = rows.iloc[0]
    for col in score_cols:
        val = row.get(col)
        feature['properties'][col] = float(val) if pd.notna(val) else 0.0

print(f"GeoJSON construit : {len(geojson_data['features'])} communes")

# ── Carte de base ─────────────────────────────────────────────────────────────
m = folium.Map(
    location=[48.8, 2.4],
    zoom_start=9,
    tiles='CartoDB positron',
    prefer_canvas=True
)

# ── Colormap (used internally only — legend is custom HTML) ───────────────────
colormap = cm.LinearColormap(
    colors=['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641'],
    vmin=0, vmax=1
)
# Note: NOT added to map — replaced by custom legend in Cell 6

# ── Couche GeoJSON principale ─────────────────────────────────────────────────
folium.GeoJson(
    geojson_data,
    name='IPS Index',
    style_function=lambda feature: {
        'fillColor': colormap(
            feature['properties'].get('eai_score', 0) or 0
        ),
        'color': 'white',
        'weight': 0.5,
        'fillOpacity': 0.75,
    },
    highlight_function=lambda feature: {
        'weight': 2,
        'color': '#333',
        'fillOpacity': 0.9,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['nom_com', 'eai_score', 'eai_rank'],
        aliases=['Commune', "Score d'accessibilité", 'Rang'],
        style="font-family:Arial;font-size:12px"
    )
).add_to(m)

folium.LayerControl(collapsed=True, position='topright').add_to(m)

print("Carte assemblée")

Construction de la carte...
GeoJSON construit : 1266 communes
Carte assemblée


In [57]:
import base64

# ── Popups riches ─────────────────────────────────────────────────────────────
popup_js_entries = []
for insee, html in popup_lookup.items():
    encoded = base64.b64encode(html.encode('utf-8')).decode('ascii')
    popup_js_entries.append(f'"{insee}": "{encoded}"')

popup_js = "var popupData = {" + ",\n".join(popup_js_entries) + "};"

popup_script = f"""
<script>
{popup_js}

document.addEventListener('DOMContentLoaded', function() {{
    setTimeout(function() {{
        {m.get_name()}.eachLayer(function(layer) {{
            if (layer.eachLayer) {{
                layer.eachLayer(function(sublayer) {{
                    if (sublayer.feature) {{
                        sublayer.off('click');
                        sublayer.on('click', function(e) {{
                            var insee = sublayer.feature.properties.insee_com;
                            var encoded = popupData[insee];
                            if (encoded) {{
                                var html = decodeURIComponent(escape(atob(encoded)));
                                L.popup({{
                                    maxWidth: 340,
                                    minWidth: 300,
                                    autoPan: true
                                }})
                                .setLatLng(e.latlng)
                                .setContent(html)
                                .openOn({m.get_name()});
                            }}
                        }});
                    }}
                }});
            }}
        }});
    }}, 1500);
}});
</script>
"""
m.get_root().html.add_child(folium.Element(popup_script))

# ── Sélecteur de catégorie ────────────────────────────────────────────────────
dropdown_options = "\n".join([
    f'<option value="{col}"{"selected" if col == "eai_score" else ""}>{label}</option>'
    for col, label in LAYERS.items()
])

js_lookups = {}
for col in score_cols:
    js_lookups[col] = {
        f['properties']['insee_com']: f['properties'][col]
        for f in geojson_data['features']
    }

js_lookup_str = "var scoreLookups = " + json.dumps(js_lookups) + ";"

colorscale_js = """
function getColor(score) {
    if (score === null || score === undefined) return '#cccccc';
    if (score >= 0.875) return '#1a9641';
    if (score >= 0.750) return '#5aae61';
    if (score >= 0.625) return '#a6d96a';
    if (score >= 0.500) return '#d9ef8b';
    if (score >= 0.375) return '#ffffbf';
    if (score >= 0.250) return '#fdae61';
    if (score >= 0.125) return '#f46d43';
    return '#d7191c';
}
"""

dropdown_html = f"""
<div id="layer-selector" style="
    position: fixed;
    top: 10px;
    left: 50px;
    z-index: 1000;
    background: white;
    padding: 10px 14px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    font-size: 13px;
">
    <label style="font-weight:bold;color:#2c3e50;display:block;margin-bottom:6px">
        🗺️ Sélectionnez une catégorie
    </label>
    <select id="layer-dropdown"
            style="width:240px;padding:4px;border:1px solid #ccc;border-radius:4px"
            onchange="switchLayer(this.value)">
        {dropdown_options}
    </select>
    <div style="font-size:10px;color:#999;margin-top:4px">
        Cliquez sur une commune pour les détails
    </div>
</div>

<script>
{js_lookup_str}

{colorscale_js}

function switchLayer(selectedCol) {{
    var lookup = scoreLookups[selectedCol];
    if (!lookup) return;

    {m.get_name()}.eachLayer(function(layer) {{
        if (layer.eachLayer) {{
            layer.eachLayer(function(sublayer) {{
                if (sublayer.feature) {{
                    var insee = sublayer.feature.properties.insee_com;
                    var score = lookup[insee];
                    if (score !== undefined) {{
                        sublayer.setStyle({{
                            fillColor: getColor(score),
                            fillOpacity: 0.75
                        }});
                    }}
                }}
            }});
        }}
    }});
}}
</script>
"""
m.get_root().html.add_child(folium.Element(dropdown_html))
print("Sélecteur et popups ajoutés")

Sélecteur et popups ajoutés


In [58]:
# ── Dropdown for dimension switching ─────────────────────────────────────────
dropdown_options = "\n".join([
    f'<option value="{col}"{"selected" if col == "eai_score" else ""}>{label}</option>'
    for col, label in LAYERS.items()
])

# Build JS score lookup for all dimensions
js_lookups = {}
for col in score_cols:
    js_lookups[col] = {
        f['properties']['insee_com']: f['properties'][col]
        for f in geojson_data['features']
    }

js_lookup_str = "var scoreLookups = " + json.dumps(js_lookups) + ";"

# Color function matching our Python colormap
colorscale_js = """
function getColor(score) {
    if (score === null || score === undefined) return '#cccccc';
    if (score >= 0.875) return '#1a9641';
    if (score >= 0.750) return '#5aae61';
    if (score >= 0.625) return '#a6d96a';
    if (score >= 0.500) return '#d9ef8b';
    if (score >= 0.375) return '#ffffbf';
    if (score >= 0.250) return '#fdae61';
    if (score >= 0.125) return '#f46d43';
    return '#d7191c';
}
"""

dropdown_html = f"""
<div id="layer-selector" style="
    position: fixed;
    top: 10px;
    left: 50px;
    z-index: 1000;
    background: white;
    padding: 10px 14px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    font-size: 13px;
">
    <label style="font-weight:bold;color:#2c3e50;display:block;margin-bottom:6px">
        🗺️ Select Index Layer
    </label>
    <select id="layer-dropdown"
            style="width:240px;padding:4px;border:1px solid #ccc;border-radius:4px"
            onchange="switchLayer(this.value)">
        {dropdown_options}
    </select>
    <div style="font-size:10px;color:#999;margin-top:4px">
        Click communes for full details
    </div>
</div>

<script>
{js_lookup_str}

{colorscale_js}

function switchLayer(selectedCol) {{
    var lookup = scoreLookups[selectedCol];
    if (!lookup) return;

    {m.get_name()}.eachLayer(function(layer) {{
        if (layer.eachLayer) {{
            layer.eachLayer(function(sublayer) {{
                if (sublayer.feature) {{
                    var insee = sublayer.feature.properties.insee_com;
                    var score = lookup[insee];
                    if (score !== undefined) {{
                        sublayer.setStyle({{
                            fillColor: getColor(score),
                            fillOpacity: 0.75
                        }});
                    }}
                }}
            }});
        }}
    }});
}}
</script>
"""

m.get_root().html.add_child(folium.Element(dropdown_html))
print("Dropdown added")

Dropdown added


In [59]:
# ── Légende ───────────────────────────────────────────────────────────────────
legend_html = """
<div style="
    position: fixed;
    top: 80px;
    right: 10px;
    z-index: 1000;
    background: white;
    padding: 14px 16px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    width: 200px;
">
    <div style="font-size:12px;font-weight:bold;color:#2c3e50;margin-bottom:10px">
        📊 Score IPS
    </div>
    <div style="
        height: 16px;
        border-radius: 4px;
        background: linear-gradient(to right, #d7191c, #fdae61, #ffffbf, #a6d96a, #1a9641);
        margin-bottom: 4px;
    "></div>
    <div style="display:flex;justify-content:space-between;
                font-size:10px;color:#666;margin-bottom:12px">
        <span>0,0</span>
        <span>0,25</span>
        <span>0,5</span>
        <span>0,75</span>
        <span>1,0</span>
    </div>
    <div style="font-size:10px;line-height:1.9">
        <div style="display:flex;align-items:center;gap:8px">
            <div style="width:14px;height:14px;border-radius:3px;
                        background:#1a9641;flex-shrink:0"></div>
            <span>Très bien équipée</span>
        </div>
        <div style="display:flex;align-items:center;gap:8px">
            <div style="width:14px;height:14px;border-radius:3px;
                        background:#a6d96a;flex-shrink:0"></div>
            <span>Bien équipée</span>
        </div>
        <div style="display:flex;align-items:center;gap:8px">
            <div style="width:14px;height:14px;border-radius:3px;
                        background:#ffffbf;flex-shrink:0;border:1px solid #ccc"></div>
            <span>Équipement moyen</span>
        </div>
        <div style="display:flex;align-items:center;gap:8px">
            <div style="width:14px;height:14px;border-radius:3px;
                        background:#fdae61;flex-shrink:0"></div>
            <span>Sous-équipée</span>
        </div>
        <div style="display:flex;align-items:center;gap:8px">
            <div style="width:14px;height:14px;border-radius:3px;
                        background:#d7191c;flex-shrink:0"></div>
            <span>Très sous-équipée</span>
        </div>
    </div>
    <div style="margin-top:10px;padding-top:8px;border-top:1px solid #eee;
                font-size:9px;color:#aaa;line-height:1.4">
        Rang percentile parmi<br>les 1 266 communes d'IDF
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ── Note méthodologique ───────────────────────────────────────────────────────
title_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 10px;
    z-index: 1000;
    background: white;
    padding: 12px 16px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    max-width: 280px;
">
    <div style="font-size:14px;font-weight:bold;color:#2c3e50;margin-bottom:4px">
        Indice de proximité aux services
    </div>
    <div style="font-size:11px;color:#555;margin-bottom:6px">
        Île-de-France &nbsp;·&nbsp; 1 266 communes
    </div>
    <div style="font-size:10px;color:#888;line-height:1.4">
        Scores calculés sur 60 types d'équipements regroupés
        en 11 dimensions. Sources : INSEE BPE 2024,
        Filosofi 2021, RP 2022, grille d'accessibilité 2023.
    </div>
    <div style="font-size:10px;color:#888;margin-top:6px;line-height:1.4">
        <b>Méthode :</b> Rang percentile normalisé.
        1,0 = commune la mieux équipée d'IDF.
        Culture et enseignement supérieur : médiane pondérée.
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))
print("Légende et titre ajoutés")

Légende et titre ajoutés


In [60]:
output_path = r"C:\Users\admin\work\idf_project\outputs\idf_eai_map.html"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

html_content = m.get_root().render()
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Carte sauvegardée : {output_path}")
print(f"Taille : {os.path.getsize(output_path)/1024/1024:.1f} Mo")

Carte sauvegardée : C:\Users\admin\work\idf_project\outputs\idf_eai_map.html
Taille : 42.8 Mo


In [74]:
# ANALYSIS REDUCING FILE SIZE FOR SERVICES POINTS 

import os
import json

geojson_output_dir = r"C:\Users\admin\work\idf_project\outputs\facilities"
os.makedirs(geojson_output_dir, exist_ok=True)

print("Exporting facility GeoJSON files...")

for group_name, indicators in FACILITY_GROUPS.items():
    # Filter facilities for this group
    group_facilities = facilities_gdf[
        facilities_gdf['indicator'].isin(indicators)
    ].copy()

    # Build clean GeoJSON
    features = []
    for _, fac in group_facilities.iterrows():
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [fac.geometry.x, fac.geometry.y]
            },
            "properties": {
                "label": fac['typequ_label'],
                "indicator": fac['indicator'],
                "commune": fac['insee_com']
            }
        })

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    # Clean filename — remove accents and spaces
    safe_name = group_name.lower()\
        .replace('é', 'e').replace('è', 'e').replace('ê', 'e')\
        .replace('à', 'a').replace('â', 'a').replace('î', 'i')\
        .replace(' ', '_').replace('—', '').replace('-', '_')\
        .replace('__', '_').strip('_')

    filepath = os.path.join(geojson_output_dir, f"{safe_name}.geojson")

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, ensure_ascii=False)

    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {group_name}: {len(features):,} points → {size_kb:.0f} KB")

print("\nDone. Files saved to outputs/facilities/")

Exporting facility GeoJSON files...
  Soins essentiels: 12,766 points → 2528 KB
  Soins complémentaires: 18,984 points → 3749 KB
  Éducation: 12,023 points → 2488 KB
  Alimentaire: 18,476 points → 3721 KB
  Transports: 429 points → 96 KB
  Services sociaux: 11,333 points → 2471 KB
  Sports: 11,764 points → 2423 KB
  Culture: 1,819 points → 344 KB
  Sécurité: 392 points → 72 KB

Done. Files saved to outputs/facilities/


In [76]:
# ── Cell 9 — Optimized facilities map ────────────────────────────────────────
import folium
import json
import os

print("Construction de la carte équipements...")

m2 = folium.Map(
    location=[48.8, 2.4],
    zoom_start=9,
    tiles='CartoDB positron',
    prefer_canvas=True
)

# Commune boundaries for context
folium.GeoJson(
    geojson_data,
    name='Communes',
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#aaa',
        'weight': 0.5,
        'fillOpacity': 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['nom_com'],
        aliases=['Commune :'],
        style="font-family:Arial;font-size:12px"
    )
).add_to(m2)

# Map group names to filenames
GROUP_FILES = {
    'Soins essentiels':       'soins_essentiels',
    'Soins complémentaires':  'soins_complementaires',
    'Éducation':              'education',
    'Alimentaire':            'alimentaire',
    'Transports':             'transports',
    'Services sociaux':       'services_sociaux',
    'Sports':                 'sports',
    'Culture':                'culture',
    'Sécurité':               'securite',
}

GROUP_COLORS = {
    'Soins essentiels':       '#e74c3c',
    'Soins complémentaires':  '#e91e8c',
    'Éducation':              '#2980b9',
    'Alimentaire':            '#e67e22',
    'Transports':             '#1a237e',
    'Services sociaux':       '#8e44ad',
    'Sports':                 '#27ae60',
    'Culture':                '#922b21',
    'Sécurité':               '#7f8c8d',
}

group_configs = []
for group_name, safe_name in GROUP_FILES.items():
    group_configs.append({
        'name': group_name,
        'file': f'facilities/{safe_name}.geojson',
        'color': GROUP_COLORS.get(group_name, '#333333')
    })

group_configs_js = json.dumps(group_configs, ensure_ascii=False)

# Get map variable name
map_var = m2.get_name()
print(f"Map variable: {map_var}")

# Title
title2_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 10px;
    z-index: 1000;
    background: white;
    padding: 12px 16px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif;
    max-width: 260px;
">
    <div style="font-size:14px;font-weight:bold;color:#2c3e50;margin-bottom:4px">
        Équipements et services — IDF
    </div>
    <div style="font-size:11px;color:#555;margin-bottom:4px">
        89 208 équipements · 60 types
    </div>
    <div style="font-size:10px;color:#888;line-height:1.4">
        Les équipements se chargent à la demande.
        Les points se regroupent selon le zoom.
        Zoom niveau 15 pour les points individuels.
    </div>
    <div style="font-size:10px;color:#888;margin-top:4px">
        Source : INSEE BPE 2024
    </div>
</div>
"""
m2.get_root().html.add_child(folium.Element(title2_html))

# Save base map first
facilities_output = r"C:\Users\admin\work\idf_project\outputs\idf_facilities_map.html"
html_content2 = m2.get_root().render()

# Inject dynamic script at end of body — AFTER all Folium scripts
# This guarantees Leaflet is fully loaded
dynamic_script = f"""
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.5.3/MarkerCluster.css"/>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.5.3/MarkerCluster.Default.css"/>
<script>
var groupConfigs = {group_configs_js};
var loadedLayers = {{}};
var layerGroups = {{}};

function loadScript(url, callback) {{
    var script = document.createElement('script');
    script.src = url;
    script.onload = callback;
    document.body.appendChild(script);
}}

function initFacilitiesMap() {{
    var map = {map_var};

    var controlDiv = L.control({{position: 'topright'}});
    controlDiv.onAdd = function(map) {{
        var div = L.DomUtil.create('div', '');
        div.style.cssText = 'background:white;padding:12px 16px;border-radius:8px;' +
            'box-shadow:0 2px 8px rgba(0,0,0,0.3);font-family:Arial,sans-serif;' +
            'font-size:12px;max-width:220px;max-height:80vh;overflow-y:auto;';

        div.innerHTML =
            '<div style="font-weight:bold;color:#2c3e50;margin-bottom:10px;font-size:13px">' +
            '\\uD83D\\uDCCD Équipements et services</div>' +
            '<div style="font-size:10px;color:#999;margin-bottom:10px;line-height:1.4">' +
            'Cochez une catégorie pour afficher les équipements sur la carte.</div>';

        groupConfigs.forEach(function(config) {{
            var row = document.createElement('div');
            row.style.cssText = 'display:flex;align-items:center;gap:8px;margin-bottom:7px';

            var checkbox = document.createElement('input');
            checkbox.type = 'checkbox';
            checkbox.id = 'cb_' + config.name;
            checkbox.style.cursor = 'pointer';

            var dot = document.createElement('div');
            dot.style.cssText = 'width:10px;height:10px;border-radius:50%;' +
                'background:' + config.color + ';flex-shrink:0';

            var label = document.createElement('label');
            label.htmlFor = 'cb_' + config.name;
            label.style.cssText = 'cursor:pointer;font-size:11px;color:#444';
            label.textContent = config.name;

            checkbox.addEventListener('change', function() {{
                if (this.checked) {{
                    loadFacilityGroup(config, map);
                }} else {{
                    if (layerGroups[config.name]) {{
                        map.removeLayer(layerGroups[config.name]);
                    }}
                }}
            }});

            row.appendChild(checkbox);
            row.appendChild(dot);
            row.appendChild(label);
            div.appendChild(row);
        }});

        L.DomEvent.disableClickPropagation(div);
        L.DomEvent.disableScrollPropagation(div);
        return div;
    }};
    controlDiv.addTo(map);
    console.log('Carte équipements initialisée');
}}

function loadFacilityGroup(config, map) {{
    var cb = document.getElementById('cb_' + config.name);
    if (cb) cb.parentElement.style.opacity = '0.5';

    if (loadedLayers[config.name]) {{
        layerGroups[config.name].addTo(map);
        if (cb) cb.parentElement.style.opacity = '1';
        return;
    }}

    fetch(config.file)
        .then(function(response) {{
            if (!response.ok) throw new Error('HTTP ' + response.status);
            return response.json();
        }})
        .then(function(data) {{
            var cluster = L.markerClusterGroup({{
                maxClusterRadius: 50,
                disableClusteringAtZoom: 15,
                chunkedLoading: true
            }});

            L.geoJSON(data, {{
                pointToLayer: function(feature, latlng) {{
                    return L.circleMarker(latlng, {{
                        radius: 4,
                        fillColor: config.color,
                        color: config.color,
                        weight: 1,
                        opacity: 0.8,
                        fillOpacity: 0.8
                    }});
                }},
                onEachFeature: function(feature, layer) {{
                    if (feature.properties && feature.properties.label) {{
                        layer.bindTooltip(feature.properties.label);
                    }}
                }}
            }}).addTo(cluster);

            loadedLayers[config.name] = true;
            layerGroups[config.name] = cluster;
            cluster.addTo(map);
            if (cb) cb.parentElement.style.opacity = '1';
            console.log('Chargé : ' + config.name + ' — ' + data.features.length + ' points');
        }})
        .catch(function(err) {{
            console.error('Erreur : ' + config.name + ' — ' + err.message);
            if (cb) {{ cb.checked = false; cb.parentElement.style.opacity = '1'; }}
        }});
}}

// Load MarkerCluster after page is ready, then init map
window.addEventListener('load', function() {{
    loadScript(
        'https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.5.3/leaflet.markercluster.js',
        initFacilitiesMap
    );
}});
</script>
"""

# Inject before </body> tag — guarantees it runs after all Folium scripts
html_content2 = html_content2.replace('</body>', dynamic_script + '\n</body>')

with open(facilities_output, 'w', encoding='utf-8') as f:
    f.write(html_content2)

size_mb = os.path.getsize(facilities_output) / 1024 / 1024
print(f"Sauvegardé : {facilities_output}")
print(f"Taille : {size_mb:.1f} Mo")

Construction de la carte équipements...
Map variable: map_a63d7f3f20953ae8451ca15cad087f5b
Sauvegardé : C:\Users\admin\work\idf_project\outputs\idf_facilities_map.html
Taille : 27.3 Mo


In [71]:
import os
size = os.path.getsize(r"C:\Users\admin\work\idf_project\outputs\idf_facilities_map.html") / 1024 / 1024
print(f"Taille : {size:.1f} Mo")

Taille : 27.3 Mo


In [77]:
eai_size = os.path.getsize(r"C:\Users\admin\work\idf_project\outputs\idf_eai_map.html") / 1024 / 1024
fac_size = os.path.getsize(r"C:\Users\admin\work\idf_project\outputs\idf_facilities_map.html") / 1024 / 1024
print(f"Carte IPS :          {eai_size:.1f} Mo")
print(f"Carte équipements :  {fac_size:.1f} Mo")
print(f"GeoJSON équipements: ~17.9 Mo (9 fichiers)")
print(f"Total :              {eai_size + fac_size + 17.9:.1f} Mo")

Carte IPS :          42.8 Mo
Carte équipements :  27.3 Mo
GeoJSON équipements: ~17.9 Mo (9 fichiers)
Total :              87.9 Mo
